In [0]:
CREATE TABLE events (
    event_id    INT        NOT NULL,
    user_id     INT        NOT NULL,
    event_time  TIMESTAMP   NOT NULL,
    event_type  VARCHAR(50),
    page        VARCHAR(100)
);
INSERT INTO events (event_id, user_id, event_time, event_type, page) VALUES
(1,  101, '2026-07-17 10:00:00', 'page_view',   'home'),
(2,  101, '2026-07-17 10:05:00', 'page_view',   'product'),
(3,  101, '2026-07-17 10:45:00', 'add_to_cart', 'product'),
(4,  101, '2026-07-17 11:10:00', 'purchase',    'checkout'),

(5,  102, '2026-07-17 09:55:00', 'page_view',   'home'),
(6,  102, '2026-07-17 10:10:00', 'page_view',   'product'),
(7,  102, '2026-07-17 10:18:00', 'page_view',   'product'),
(8,  102, '2026-07-17 11:00:00', 'add_to_cart', 'product'),

(9,  103, '2026-07-17 10:00:00', 'page_view',   'home'),
(10, 103, '2026-07-17 10:00:00', 'page_view',   'home'),  -- deliberate duplicate
(11, 103, '2026-07-17 10:30:00', 'page_view',   'product');

WITH events_with_diff AS (
    SELECT
        e.*,
        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS prev_event_time,
        DATEDIFF(
            MINUTE,
            LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
            event_time
        ) AS diff_minutes
    FROM events e
)
SELECT *
FROM events_with_diff
ORDER BY user_id, event_time;

WITH events_with_flags AS (
    SELECT
        e.*,
        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS prev_event_time,
        DATEDIFF(
            MINUTE,
            LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
            event_time
        ) AS diff_minutes,
        CASE
            WHEN LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL
                THEN 1   -- first event for user
            WHEN DATEDIFF(
                     MINUTE,
                     LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
                     event_time
                 ) >= 30
                THEN 1   -- gap >= 30 minutes -> new session
            ELSE 0
        END AS new_session_flag
    FROM events e
)
SELECT *
FROM events_with_flags
ORDER BY user_id, event_time;

WITH events_with_flags AS (
    SELECT
        e.*,
        CASE
            WHEN LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL
                THEN 1
            WHEN DATEDIFF(
                     MINUTE,
                     LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
                     event_time
                 ) >= 30
                THEN 1
            ELSE 0
        END AS new_session_flag
    FROM events e
),
events_with_session AS (
    SELECT
        *,
        SUM(new_session_flag) OVER (
            PARTITION BY user_id
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS session_number
    FROM events_with_flags
)
SELECT
    event_id,
    user_id,
    event_time,
    event_type,
    page,
    CONCAT(
        CAST(user_id AS VARCHAR(10)),
        '_',
        CAST(session_number AS VARCHAR(10))
    ) AS session_id
FROM events_with_session
ORDER BY user_id, event_time;
--dedup
WITH ranked_events AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, event_time, event_type, page
            ORDER BY event_id
        ) AS rn
    FROM events
)
SELECT *
FROM ranked_events
ORDER BY user_id, event_time, event_id;
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, event_time, event_type, page
            ORDER BY event_id
        ) AS rn
    FROM events
) t
WHERE rn = 1;

--inspect deduplicates
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, event_time, event_type, page
            ORDER BY event_id
        ) AS rn
    FROM events
) t
WHERE rn > 1;
--top(n)

WITH events_with_session AS (
    SELECT
        e.*,
        CASE
            WHEN LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL
                THEN 1
            WHEN DATEDIFF(
                     MINUTE,
                     LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
                     event_time
                 ) >= 30
                THEN 1
            ELSE 0
        END AS new_session_flag
    FROM events e
),
sessionized AS (
    SELECT
        *,
        SUM(new_session_flag) OVER (
            PARTITION BY user_id
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS session_number
    FROM events_with_session
)
WITH events_with_session AS (  CREATE OR REPLACE TABLE session_events AS
SELECT
    event_id,
    user_id,
    event_time,
    event_type,
    page,
    CONCAT(user_id, '_', session_number) AS session_id
FROM events_with_session;